Name: Muhammad Raziq Bin Sufian

Id: 24006626

## 1) Data Format & Digital Twin Relevance

**Format chosen: JSON.** It's lightweight, language-independent, and natively supported by both HTTP REST and MQTT. Every message shares one envelope, `{source_id, data_type, timestamp, payload}`, so both protocols can be parsed and validated the same way regardless of what's inside payload.

**Two data types, both relevant to a digital twin:**
- **Numbers — environmental sensors** (temperature, humidity, power draw). Represents periodic condition-monitoring of a physical asset that the twin needs to mirror.
- **Coordinates — GPS telemetry** (latitude, longitude, speed). Represents the continuously-changing position of a moving physical asset that the twin needs to track in real time.


## 2) HTTP REST — Justification, Route, Trigger & Completion

**Justification:** The environmental readings are periodic, independent snapshots, not a continuous stream. Each one can be sent as its own self-contained request with no need to keep a connection open. REST's stateless request/response model also makes each update easy to validate and retry individually if one fails, since it doesn't depend on any earlier message.

**Protocol route/interface:** A local Flask server acts as the digital twin's ingestion endpoint at `http://127.0.0.1:5000/api/twin/update`. It runs on a background thread so it can accept requests while the rest of the notebook keeps running. Sensor sources send `POST` requests with a JSON body to this endpoint.

**Trigger:** a new request is sent every time a sensor source has a fresh reading ready (simulated here by calling `generate_number_data()`).

**Completion:** the server parses and stores the payload, then replies `200 OK` with a JSON acknowledgement; the request is considered complete when the sender's `requests.post()` call returns.

## 3) MQTT — Justification, Route, Trigger & Completion

**Justification:** GPS position changes continuously, so opening and closing a new HTTP connection for every update would be wasteful. MQTT keeps one lightweight, persistent connection to a broker and lets any number of updates flow over it as a stream, which fits continuous telemetry much better than request/response.

**Protocol route/interface:** the public broker `broker.hivemq.com` on port `1883`, topic `utp/digitaltwin/telemetry/coordinates`. `loop_start()` runs the network loop on a background thread so publishing/receiving doesn't block the notebook.

**Trigger:** each coordinate source publishes to the topic whenever it has a new position (simulated here by calling `generate_coordinate_data()`).

**Completion:** for the sender, publishing completes once `client.publish()` hands the message to the client's outgoing queue, for the digital twin side, the update is considered received once the subscriber's `on_message` callback fires and stores the payload — which is what we verify in section 6.

## 4) Install dependencies

In [1]:
%pip install flask requests
%pip install paho-mqtt

Defaulting to user installation because normal site-packages is not writeableNote: you may need to restart the kernel to use updated packages.

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


## 5) Generate digital twin data (numbers & coordinates)

In [ ]:
import random
import time
import json
from datetime import datetime

def generate_number_data(source_id):
    return {
        "source_id": source_id,
        "data_type": "Environmental_Numbers",
        "timestamp": datetime.now().isoformat(),
        "payload": {
            "temperature_c": round(random.uniform(22.0, 28.0), 2),
            "humidity_pct": round(random.uniform(45.0, 55.0), 2),
            "power_usage_kw": round(random.uniform(1.2, 3.8), 2)
        }
    }

def generate_coordinate_data(source_id):
    base_lat = 4.385
    base_lon = 100.979
    return {
        "source_id": source_id,
        "data_type": "GPS_Coordinates",
        "timestamp": datetime.now().isoformat(),
        "payload": {
            "latitude": round(base_lat + random.uniform(-0.005, 0.005), 6),
            "longitude": round(base_lon + random.uniform(-0.005, 0.005), 6),
            "speed_kmh": round(random.uniform(0.0, 15.0), 1)
        }
    }

#  check
print("=== Numbers ===")
print(json.dumps(generate_number_data("Sensor_A"), indent=2))

print("\n=== Coordinates ===")
print(json.dumps(generate_coordinate_data("Bot_1"), indent=2))

=== Numbers ===
{
  "source_id": "Sensor_A",
  "data_type": "Environmental_Numbers",
  "timestamp": "2026-07-05T00:07:35.531409",
  "payload": {
    "temperature_c": 26.37,
    "humidity_pct": 45.48,
    "power_usage_kw": 3.07
  }
}

=== Coordinates ===
{
  "source_id": "Bot_1",
  "data_type": "GPS_Coordinates",
  "timestamp": "2026-07-05T00:07:35.532450",
  "payload": {
    "latitude": 4.386491,
    "longitude": 100.982961,
    "speed_kmh": 8.2
  }
}


## 6) HTTP REST endpoint (receiver)

In [4]:
import threading
from flask import Flask, request, jsonify

app = Flask(__name__)
http_received_storage = []

@app.route('/api/twin/update', methods=['POST'])
def receive_twin_update():
    if not request.is_json:
        return jsonify({"error": "Payload must be JSON"}), 400
    data = request.get_json()
    http_received_storage.append(data)
    print(f"[HTTP SERVER] Received update from {data.get('source_id')}: {data}")
    return jsonify({"status": "success"}), 200

def start_flask_server():
    app.run(host='127.0.0.1', port=5000, debug=False, use_reloader=False, threaded=True)

server_thread = threading.Thread(target=start_flask_server, daemon=True)
server_thread.start()
time.sleep(1)

print("HTTP server ready at http://127.0.0.1:5000/api/twin/update")

 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit


HTTP server ready at http://127.0.0.1:5000/api/twin/update


## 7) MQTT endpoint (subscriber)

In [ ]:
import paho.mqtt.client as mqtt

mqtt_received_storage = []

MQTT_BROKER = "broker.hivemq.com"
MQTT_PORT = 1883
MQTT_TOPIC = "utp/digitaltwin/telemetry/coordinates"

def on_connect(client, userdata, flags, reason_code, properties):
    print(f"[MQTT] Connected with code {reason_code}")
    client.subscribe(MQTT_TOPIC)

def on_message(client, userdata, msg):
    payload = json.loads(msg.payload.decode('utf-8'))
    mqtt_received_storage.append(payload)
    print(f"[MQTT SUBSCRIBER] Received stream update from {payload.get('source_id')}: {payload}")

mqtt_client = mqtt.Client(mqtt.CallbackAPIVersion.VERSION2, client_id="utp_dt_client")
mqtt_client.on_connect = on_connect
mqtt_client.on_message = on_message
mqtt_client.connect(MQTT_BROKER, MQTT_PORT)
mqtt_client.loop_start()

time.sleep(2)  

[MQTT] Connected with code Success


## 8) Parallel transmission from 2 sources per protocol + verification

Sends two HTTP sources (`Sensor_A`, `Sensor_B`) and two MQTT sources (`Bot_1`, `Bot_2`) at the same time using a thread pool, then checks that every message sent was received intact by the corresponding side.

In [ ]:
import concurrent.futures
import requests

http_received_storage.clear()
mqtt_received_storage.clear()

# 2 sources for HTTP REST
http_payload_1 = generate_number_data("Sensor_A")
http_payload_2 = generate_number_data("Sensor_B")

# 2 sources for MQTT
mqtt_payload_1 = generate_coordinate_data("Bot_1")
mqtt_payload_2 = generate_coordinate_data("Bot_2")

def send_http(payload):
    requests.post("http://127.0.0.1:5000/api/twin/update", json=payload)

def send_mqtt(payload):
    mqtt_client.publish(MQTT_TOPIC, json.dumps(payload))

print("=== Sending data in parallel ===")
with concurrent.futures.ThreadPoolExecutor(max_workers=4) as executor:
    futures = [
        executor.submit(send_http, http_payload_1),
        executor.submit(send_http, http_payload_2),
        executor.submit(send_mqtt, mqtt_payload_1),
        executor.submit(send_mqtt, mqtt_payload_2),
    ]
    for f in futures:
        f.result()

time.sleep(5)  

print("\n=== Verifying received data ===")
assert http_payload_1 in http_received_storage, "HTTP Source 1 missing/corrupted!"
assert http_payload_2 in http_received_storage, "HTTP Source 2 missing/corrupted!"
print("HTTP REST: both asynchronous updates received correctly.")

assert mqtt_payload_1 in mqtt_received_storage, "MQTT Source 1 missing/corrupted!"
assert mqtt_payload_2 in mqtt_received_storage, "MQTT Source 2 missing/corrupted!"
print("MQTT: both streamed updates received correctly.")

print("\nAll data transmitted and verified successfully.")

127.0.0.1 - - [05/Jul/2026 00:12:55] "POST /api/twin/update HTTP/1.1" 200 -
127.0.0.1 - - [05/Jul/2026 00:12:55] "POST /api/twin/update HTTP/1.1" 200 -


=== Sending data in parallel ===
[HTTP SERVER] Received update from Sensor_B: {'source_id': 'Sensor_B', 'data_type': 'Environmental_Numbers', 'timestamp': '2026-07-05T00:12:55.204099', 'payload': {'temperature_c': 26.24, 'humidity_pct': 47.53, 'power_usage_kw': 1.63}}
[HTTP SERVER] Received update from Sensor_A: {'source_id': 'Sensor_A', 'data_type': 'Environmental_Numbers', 'timestamp': '2026-07-05T00:12:55.204099', 'payload': {'temperature_c': 27.02, 'humidity_pct': 52.12, 'power_usage_kw': 3.57}}
[MQTT SUBSCRIBER] Received stream update from Bot_1: {'source_id': 'Bot_1', 'data_type': 'GPS_Coordinates', 'timestamp': '2026-07-05T00:12:55.204099', 'payload': {'latitude': 4.389216, 'longitude': 100.975865, 'speed_kmh': 13.5}}
[MQTT SUBSCRIBER] Received stream update from Bot_2: {'source_id': 'Bot_2', 'data_type': 'GPS_Coordinates', 'timestamp': '2026-07-05T00:12:55.204099', 'payload': {'latitude': 4.38335, 'longitude': 100.982854, 'speed_kmh': 10.9}}

=== Verifying received data ===
HTT